In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import subprocess; subprocess.run(['pip', 'install', '-q', '--upgrade', 'openpyxl'])

import time
import requests
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY          = 'neom-9172-sjou-mzxw'
BASE_URL         = 'https://api.boxtobox.ai/v1/wy/api/v3'
COMPETITION_WYID = 364
SEASON_ID        = 191622
SLEEP_SECONDS    = 0.15
OUT_XLSX         = '/content/drive/MyDrive/Event data/Wyscout/epl.xlsx'

def api(path, **params):
    r = requests.get(f'{BASE_URL}{path}', params={'api_key': API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

# ── Step 1: get all matches in the season ────────────────────────────────────
print('Fetching match list...')
resp    = api(f'/competitions/{COMPETITION_WYID}/matches', seasonId=SEASON_ID)
matches = resp.get('matches', resp) if isinstance(resp, dict) else resp
match_ids = [m['wyId'] for m in matches]
print(f'  {len(match_ids)} matches found')

# ── Step 2: fetch /matches/{wyId}/advancedstats/players for every match ───────
rows, failures = [], []

for i, mid in enumerate(match_ids, 1):
    print(f'[{i}/{len(match_ids)}] match {mid}', end='\r')
    try:
        data = api(f'/matches/{mid}/advancedstats/players')
        players = data.get('players', data) if isinstance(data, dict) else data
        for entry in players:
            flat = pd.json_normalize(entry, sep='.')
            flat['matchId'] = mid
            rows.append(flat)
    except Exception as e:
        failures.append({'matchId': mid, 'error': str(e)})
    time.sleep(SLEEP_SECONDS)

print(f'\nDone — {len(rows)} player-match rows, {len(failures)} failures')

# ── Step 3: build final dataframe ─────────────────────────────────────────────
df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
df_fail = pd.DataFrame(failures)

# tidy up column order: identity cols first, stats alphabetically after
FRONT = ['matchId','playerId','shortName','lastName','firstName',
         'teamId','teamName','roleCode','roleName']
front = [c for c in FRONT if c in df.columns]
rest  = sorted(c for c in df.columns if c not in front)
df    = df[front + rest]

print(f'Shape: {df.shape}')
df.head()

# ── Excel style helpers ────────────────────────────────────────────────────────
HEADER_BG     = '1F3864'
ID_BG         = 'D6E4F0'
ALT_BG        = 'F2F7FB'
ERR_BG        = 'FFF2CC'
BORDER        = Border(**{s: Side(style='thin', color='BDD7EE')
                          for s in ('left','right','top','bottom')})
IDENTITY_COLS = set(front)

def _hdr(cell, v):
    cell.value     = v
    cell.font      = Font(name='Calibri', bold=True, color='FFFFFF', size=11)
    cell.fill      = PatternFill('solid', fgColor=HEADER_BG)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border    = BORDER

def _cell(cell, v, bg, align='right', bold=False, num_fmt=None):
    cell.value     = v
    cell.fill      = PatternFill('solid', fgColor=bg)
    cell.font      = Font(name='Calibri', size=10, bold=bold)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    cell.border    = BORDER
    if num_fmt: cell.number_format = num_fmt

def _width(series, header, cap=30):
    mx = series.dropna().astype(str).str.len().max() if not series.dropna().empty else 0
    return min(max(len(str(header)), mx) + 2, cap)

def _numfmt(col, series):
    low = col.lower()
    if 'id' in low: return '0'
    if any(k in low for k in ('percent','pct','rate','ratio','avg','mean')): return '0.00'
    if pd.api.types.is_numeric_dtype(series):
        ok = series.dropna()
        if ok.empty or ok.apply(float.is_integer).all(): return '0'
        return '0.00'
    return 'General'

def fmt_stats(ws, df):
    cols = list(df.columns)
    for ci, c in enumerate(cols, 1): _hdr(ws.cell(1, ci), c)
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        alt = ri % 2 == 0
        for ci, c in enumerate(cols, 1):
            v = row[c]; v = v.item() if hasattr(v, 'item') else v
            if c in IDENTITY_COLS:
                _cell(ws.cell(ri, ci), v, ID_BG, align='left',
                      bold=(c in ('shortName','lastName')))
            else:
                _cell(ws.cell(ri, ci), v, ALT_BG if alt else 'FFFFFF',
                      num_fmt=_numfmt(c, df[c]) if pd.api.types.is_numeric_dtype(df[c]) else None)
    for ci, c in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = _width(df[c], c)
    id_n = sum(1 for c in cols if c in IDENTITY_COLS)
    ws.freeze_panes = f'{get_column_letter(id_n + 1)}2'
    ws.row_dimensions[1].height = 36
    for r in range(2, len(df) + 2): ws.row_dimensions[r].height = 16
    if len(df):
        t = Table(displayName='AdvancedStats',
                  ref=f'A1:{get_column_letter(len(cols))}{len(df)+1}')
        t.tableStyleInfo = TableStyleInfo(name='TableStyleMedium9', showRowStripes=True)
        ws.add_table(t)
    ws.title = 'Advanced Stats'

def fmt_failures(ws, df):
    for ci, c in enumerate(df.columns, 1): _hdr(ws.cell(1, ci), c)
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, c in enumerate(df.columns, 1):
            v = row[c]; v = v.item() if hasattr(v, 'item') else v
            _cell(ws.cell(ri, ci), v, ERR_BG, align='left')
    for ci, c in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = _width(df[c], c)
    ws.freeze_panes = 'A2'
    ws.title = 'Failures'

# ── Step 4: write & format Excel ──────────────────────────────────────────────
with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as w:
    df.to_excel(w, sheet_name='advancedstats', index=False)
    if not df_fail.empty: df_fail.to_excel(w, sheet_name='failures', index=False)

wb = load_workbook(OUT_XLSX)
fmt_stats(wb['advancedstats'], df)
if not df_fail.empty and 'failures' in wb.sheetnames:
    fmt_failures(wb['failures'], df_fail)
wb.save(OUT_XLSX)

print(f'Saved → {OUT_XLSX}')